# Jour 1 — Retards des vols américains en 2015

## Fiche de cadrage
- **Sujet :** Quels aéroports, compagnies et créneaux concentrent les retards ?
- **Source :** `usdot/flight-delays`, version 1 (Bureau of Transportation Statistics / Kaggle).
- **Tables :** `flights.csv` (faits), `airlines.csv` et `airports.csv` (dimensions).
- **Périmètre :** tous les vols de janvier, avril, juillet et novembre 2015. Un mois par saison météorologique ; aucune sélection selon le retard. Le total et les empreintes SHA-256 sont dans `reports/data_manifest.json`.
- **Limites :** extrait non aléatoire, ne permettant pas d'estimer une année entière. Novembre inclut Thanksgiving. Ces données historiques ne classent pas les compagnies actuelles. Octobre n'est pas retenu : la source y emploie des identifiants aéroports numériques.
- **Jointures prévues :** `flights.airline = airlines.iata_code`, `flights.origin_airport = airports.iata_code`, puis une seconde jointure pour la destination. Dimensions uniques, jointures gauches et comptage des clés non appariées au Jour 2.
- **Temps :** année/mois/jour, jour de semaine, départ prévu au format HHMM local de l'aéroport. Aucun calcul de durée entre fuseaux à partir de ces heures locales.
- **Mesures :** retard au départ/à l'arrivée (minutes), distance (miles), durée (minutes), annulation.
- **Dimensions :** compagnie, origine, destination, saison et créneau du départ prévu.

## Questions et indicateurs à construire
1. Quels aéroports concentrent le **total de minutes positives de retard au départ** ? Classement et Pareto.
2. Quels groupes ont les plus forts **taux de retard ≥ 15 min**, médiane et P90 ? Afficher les effectifs pour distinguer volume et fiabilité.
3. Matin (06:00–11:59) et soir (18:00–23:59) présentent-ils des distributions différentes ? Les autres horaires restent dans les analyses globales.
4. Quels profils d'aéroports ressortent d'un K-Means (volume, retards, annulations) ?

Les retards négatifs signifient une avance : ce ne sont pas des erreurs. Ne pas convertir les valeurs manquantes des vols annulés en retard nul. Taux d'annulation sur tous les vols ; taux de retard sur les vols non annulés dont le retard est renseigné. Les associations observées ne démontrent pas une responsabilité causale.

## Objectifs du Jour 1
Inspecter HDFS (blocs de 16 Mio, réplication 2), déposer les trois CSV, vérifier la résilience et créer les tables Delta Bronze avec traçabilité. Le nettoyage et les jointures sont réservés au Jour 2.

## 1. Données et reproductibilité
Lancer une seule fois `docker exec spark-jupyter python /workspace/scripts/prepare_flights.py` pour télécharger la source et sélectionner les quatre mois. Le script ne modifie pas le cache original Kaggle. Cette cellule vérifie les fichiers locaux sans télécharger à nouveau l'année complète.

In [1]:
import hashlib
import json
from pathlib import Path

ROOT = Path('/workspace')
DATA_RAW = ROOT / 'data_raw'
manifest = json.loads((ROOT / 'reports/data_manifest.json').read_text(encoding='utf-8'))
assert manifest['months'] == [1, 4, 7, 11]
local_files = [DATA_RAW / 'flight-delays' / f['name'] for f in manifest['files']]
for path, spec in zip(local_files, manifest['files']):
    with path.open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == spec['sha256']
    size = path.stat().st_size
    print(f'{path.name}: {size / 1024**2:.2f} Mio, {(size + 16777215) // 16777216} blocs attendus')
assert 50 * 1024**2 <= sum(p.stat().st_size for p in local_files) <= 500 * 1024**2
assert max(p.stat().st_size for p in local_files) > 32 * 1024**2
print('Vols sélectionnés :', manifest['flight_rows'])
print('Répartition mensuelle :', manifest['rows_by_month'])

airlines.csv: 0.00 Mio, 1 blocs attendus
airports.csv: 0.02 Mio, 1 blocs attendus


flights.csv: 190.05 Mio, 12 blocs attendus
Vols sélectionnés : 1943809
Répartition mensuelle : {'1': 469968, '4': 485151, '7': 520718, '11': 467972}


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName('J1_HDFS_Ingestion_Bronze')
         .master('local[2]').config('spark.sql.shuffle.partitions', '8').getOrCreate())
default_fs = spark.conf.get('spark.hadoop.fs.defaultFS')
block_size = int(spark.conf.get('spark.hadoop.dfs.blocksize'))
replication = int(spark.conf.get('spark.hadoop.dfs.replication'))
assert block_size == 16 * 1024**2 and replication == 2
print('Spark :', spark.version, '| HDFS :', default_fs)
print('Bloc :', block_size, 'octets | Réplication :', replication)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7ab72d5c-74b6-487b-97eb-e8f1e2d810e4;1.0
	confs: [default]


	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central


	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central


	found io.delta#delta-kernel-defaults;4.4.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.7-3 in central


	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central
	found io.delta#delta-kernel-unitycatalog;4.4.1 in central
:: resolution report :: resolve 832ms :: artifacts dl 22ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-databind;2.13.5 from central in [default]
	com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 from central in [default]
	com.github.luben#zstd-jni;1.5.7-3 from central in [default]
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	commons-logging#commons-logging;1.3.0 from central in [default]
	commons-pool#commons-pool;1.6 from central in [default]
	io.airlift#aircompressor;2.0.2 from central in [default]
	io.delta#delta-kernel-api;4.4.1 from central in [default]
	

26/11/16 13:17:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark : 4.2.0 | HDFS : hdfs://namenode:9000
Bloc : 16777216 octets | Réplication : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [3]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/flight-delays/airlines.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/flight-delays/airports.csv


Téléversé vers HDFS : /lakehouse/bronze/raw/flight-delays/flights.csv


In [4]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : flight-delays/flights.csv (190.05 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #5 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode1']
   Bloc #6 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #7 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #8 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #9 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #10 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #11 (14.05 Mo) -> DataNodes : ['datanode2', 'datanode1']
Fichier : flight-delays/airports.csv (0.02 Mo | repl=2)
   Bloc #0 ( 0.02 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fichier : flight-delays/airlines.csv (0.00 Mo | repl=

### Réponses aux questions sur HDFS
1. Pour un CSV non compressé de `S` octets, le nombre de blocs attendu est `ceil(S / 16 777 216)`. La cellule précédente affiche les blocs réels et les deux hôtes de chaque bloc : comparer ces résultats à l'inventaire local.
2. Un fichier de 3 Mio n'occupe pas systématiquement 16 Mio par réplique : 16 Mio est la taille maximale d'un bloc. En réplication 2, compter environ 6 Mio de données, auxquels s'ajoutent les sommes de contrôle et métadonnées.
3. Avec deux copies sur trois DataNodes, une panne simple conserve une copie lisible et laisse un troisième nœud pour reconstruire la seconde copie. Cela ne protège pas contre toutes les pannes simultanées.

## 4. Résilience et partitions
Pour réaliser et enregistrer le crash-test : `python scripts/crash_test_flights.py` **depuis Windows**. Ce script arrête `datanode2`, attend que le NameNode le déclare mort, relit le CSV avec Spark, relève les blocs, puis redémarre le nœud dans un `finally`.

Le rapport est écrit dans `reports/crash_test_flights.json`. L'arrêt des heartbeats est détecté en environ 50 s ; la reconstruction des copies peut prendre plus longtemps. Une lecture réussie avant détection ne prouve pas encore que la réplication a été restaurée.

In [5]:
print('Fichier contrôlé :', largest_hdfs_path)
df_check = spark.read.option('header', True).csv(largest_hdfs_path)
row_count = df_check.count()
assert row_count == manifest['flight_rows']
print('Lignes :', row_count)
print('Partitions Spark :', df_check.rdd.getNumPartitions())
print('Taille max de partition de lecture :', spark.conf.get('spark.sql.files.maxPartitionBytes'))
print('Les partitions Spark ne sont pas nécessairement égales aux blocs HDFS :')
print('Spark peut regrouper des blocs selon maxPartitionBytes ; CSV splittable, coûts d’ouverture et parallélisme interviennent.')

Fichier contrôlé : hdfs://namenode:9000/lakehouse/bronze/raw/flight-delays/flights.csv


Lignes : 1943809
Partitions Spark : 2
Taille max de partition de lecture : 134217728b
Les partitions Spark ne sont pas nécessairement égales aux blocs HDFS :
Spark peut regrouper des blocs selon maxPartitionBytes ; CSV splittable, coûts d’ouverture et parallélisme interviennent.


---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de vos datasets dans `/lakehouse/bronze/<nom_table>`.

In [6]:
BRONZE_SOURCES = {
    'flights': ('usdot/flight-delays', '/lakehouse/bronze/raw/flight-delays/flights.csv'),
    'airlines': ('usdot/flight-delays', '/lakehouse/bronze/raw/flight-delays/airlines.csv'),
    'airports': ('usdot/flight-delays', '/lakehouse/bronze/raw/flight-delays/airports.csv'),
}
bronze_counts = {}
for table_name, (dataset_name, raw_path) in BRONZE_SOURCES.items():
    # Bronze conserve les valeurs sources en chaînes ; typage métier au Jour 2.
    df_raw = spark.read.option('header', True).option('mode', 'FAILFAST').csv(raw_path)
    df_raw = df_raw.toDF(*[c.strip().lower().replace(' ', '_') for c in df_raw.columns])
    # Spark 4 : colonne de métadonnées du lecteur, ajoutée avant l'écriture Delta.
    df_bronze = (df_raw.withColumn('_source_file', F.col('_metadata.file_path'))
                 .withColumn('_source_dataset', F.lit(dataset_name))
                 .withColumn('_ingested_at', F.current_timestamp()))
    bronze_delta_path = f'/lakehouse/bronze/{table_name}'
    (df_bronze.write.format('delta').mode('overwrite')
     .option('overwriteSchema', 'true').save(bronze_delta_path))
    result = spark.read.format('delta').load(bronze_delta_path)
    count = result.count()
    assert result.filter(F.col('_source_file').isNull() | F.col('_ingested_at').isNull()).count() == 0
    bronze_counts[table_name] = count
    print(f'[BRONZE OK] {table_name}: {count:,} lignes')
assert bronze_counts['flights'] == manifest['flight_rows']
assert bronze_counts['airlines'] == 14
assert bronze_counts['airports'] == 322
(ROOT / 'reports/bronze_counts.json').write_text(json.dumps(bronze_counts, indent=2), encoding='utf-8')

26/11/16 13:17:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[BRONZE OK] flights: 1,943,809 lignes


[BRONZE OK] airlines: 14 lignes


[BRONZE OK] airports: 322 lignes


61

In [7]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['airlines', 'airports', 'flights']


## Suite — Jour 2
Typer les dates et les nombres, contrôler les doublons sans supprimer les vols distincts, conserver les avances négatives, mesurer les valeurs manquantes et vérifier les clés aéroports/compagnies. Créer le créneau du départ prévu et la saison, puis joindre les dimensions avec `broadcast`. Conserver la granularité : une ligne par vol.

In [8]:
spark.stop()


## Résultats du Jour 1 — vérifiés sur cette installation

- Notebook exécuté intégralement, sans erreur : **1 943 809 vols**, **14 compagnies**, **322 aéroports** en Delta Bronze.
- Fichier des vols : **199 285 860 octets**, **12 blocs HDFS** de 16 Mio au maximum, réplication **2**.
- Lecture Spark : **2 partitions**, avec `spark.sql.files.maxPartitionBytes = 128 Mio`.
- Crash-test réalisé le 2026-11-16T13:19:06.041301+00:00 : DataNode 2 déclaré mort après **60.9 s** (temps observé sur cette machine).
- Pendant la panne : **1,943,809 lignes relues**, **2 partitions**, **0 bloc sous-répliqué**, **0 bloc manquant** après réparation.
- Après redémarrage : **3 DataNodes actifs**.

La lecture reste disponible grâce aux copies survivantes. Le rapport `reports/crash_test_flights.json` conserve la chronologie JMX et les emplacements des blocs après réparation. Les sorties des cellules d'inspection précédentes décrivent l'état **avant** la panne ; le placement peut ensuite changer.
